# `shared.ipynb` — Core library (Phase A + B + helpers)

Run with `%run shared.ipynb` from the other notebooks. Defines, with no side effects:

* **Phase A** — `PathologyPatchDataset`, Albumentations builders, `HEDStainJitter`,
  `MacenkoNormalizer`, `ensure_dataset`, `make_dataloaders`.
* **Phase B** — `build_model` (timm full fine-tune **and** frozen-Phikon + MLP head),
  `get_param_groups`.
* **Helpers** — Grad-CAM (`GradCAMViT`/`GradCAMResNet`), metrics, resume-safe checkpoints.

This file deliberately does **not** download data or build models on import.

In [ ]:
# Dependency bootstrap. The container env is not persistent, so this reinstalls each session.
import importlib.util, subprocess, sys

def _ensure(mod, pkg=None):
    "Install pkg (pip name) only if module `mod` is missing. Safe to call repeatedly."
    if importlib.util.find_spec(mod) is None:
        print(f"installing {pkg or mod} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "--root-user-action=ignore", pkg or mod], check=False)

# Core libs needed to import this file and train with timm + W&B:
for _mod, _pkg in {"albumentations": "albumentations", "skimage": "scikit-image",
                   "cv2": "opencv-python-headless", "timm": "timm",
                   "sklearn": "scikit-learn", "seaborn": "seaborn",
                   "matplotlib": "matplotlib", "wandb": "wandb"}.items():
    _ensure(_mod, _pkg)
# transformers (Phikon), gradio (demo) and torchstain (Macenko) are installed on demand by
# the code that needs them, to avoid pulling heavy/conflicting deps during plain training.


In [ ]:
import os, sys, json, math, time, random, logging, zipfile, urllib.request
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("ft004")

# ---- constants -------------------------------------------------------
CLASSES = ["ADI", "BACK", "DEB", "LYM", "MUC", "MUS", "NORM", "STR", "TUM"]
CLASS_DESCRIPTIONS = {
    "ADI": "Adipose (fat tissue)",
    "BACK": "Background",
    "DEB": "Debris",
    "LYM": "Lymphocytes",
    "MUC": "Mucus",
    "MUS": "Smooth muscle",
    "NORM": "Normal colon mucosa",
    "STR": "Cancer-associated stroma",
    "TUM": "Colorectal adenocarcinoma epithelium",
}
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)


def get_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def get_amp_dtype(device):
    '''bf16 on MI300X (native, no GradScaler); fp16 on other CUDA; fp32 on CPU.'''
    if device.type != "cuda":
        return torch.float32
    if torch.cuda.is_bf16_supported():
        return torch.bfloat16
    return torch.float16


## Phase A — Augmentation (Albumentations)

`HEDStainJitter` is the stain-robustness innovation: it perturbs the Hematoxylin–Eosin–DAB
colour space (Tellez et al. 2018) to simulate inter-lab / inter-scanner stain variation —
the #1 deployment failure mode. Spatial transforms force the model onto morphology, not colour.

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2
from skimage.color import rgb2hed, hed2rgb


class HEDStainJitter(A.ImageOnlyTransform):
    '''Randomly perturb the H&E (HED) colour space. img: HxWx3 uint8 RGB.'''

    def __init__(self, theta=0.05, p=0.5):
        super().__init__(p=p)
        self.theta = theta

    def apply(self, img, **params):
        rgb = img.astype(np.float32) / 255.0
        hed = rgb2hed(rgb)
        for c in range(3):
            alpha = np.random.uniform(1 - self.theta, 1 + self.theta)
            beta = np.random.uniform(-self.theta, self.theta)
            hed[:, :, c] = hed[:, :, c] * alpha + beta
        out = np.clip(hed2rgb(hed), 0.0, 1.0)
        return (out * 255.0).astype(np.uint8)

    def get_transform_init_args_names(self):
        return ("theta",)


def build_train_aug(use_stain_aug=True, img_size=224, theta=0.05):
    tfms = [
        A.RandomRotate90(p=0.5),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.Affine(translate_percent=0.05, scale=(0.9, 1.1), rotate=(-15, 15),
                 mode=cv2.BORDER_REFLECT_101, p=0.5),
        A.OneOf([
            A.ElasticTransform(alpha=1.0, sigma=20.0, p=1.0),
            A.GridDistortion(num_steps=5, distort_limit=0.1, p=1.0),
            A.OpticalDistortion(distort_limit=0.1, p=1.0),
        ], p=0.3),
    ]
    if use_stain_aug:
        tfms += [
            HEDStainJitter(theta=theta, p=0.8),
            A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=15, val_shift_limit=10, p=0.3),
            A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=0.3),
        ]
    tfms += [
        A.Resize(img_size, img_size),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ]
    return A.Compose(tfms)


def build_eval_aug(img_size=224):
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])


def build_ood_aug(img_size=224, theta=0.15):
    '''Strong synthetic stain shift = a 'different lab' for the OOD ablation.'''
    return A.Compose([
        HEDStainJitter(theta=theta, p=1.0),
        A.HueSaturationValue(hue_shift_limit=25, sat_shift_limit=30, val_shift_limit=20, p=1.0),
        A.Resize(img_size, img_size),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])


In [ ]:
class MacenkoNormalizer:
    '''Optional Macenko stain normalization (torchstain). For OOD targets / demo uploads.'''

    def __init__(self, target_img=None):
        _ensure("torchstain")
        import torchstain
        self.normalizer = torchstain.normalizers.MacenkoNormalizer(backend="numpy")
        self.fitted = False
        if target_img is not None:
            self.fit(target_img)

    def fit(self, target_img):  # target_img: HxWx3 uint8 RGB
        self.normalizer.fit(np.asarray(target_img, dtype=np.uint8))
        self.fitted = True

    def __call__(self, img):  # img: HxWx3 uint8 RGB -> uint8 RGB
        if not self.fitted:
            return img
        try:
            norm, _, _ = self.normalizer.normalize(I=np.asarray(img, dtype=np.uint8), stains=False)
            return np.asarray(norm, dtype=np.uint8)
        except Exception as e:
            logger.warning(f"Macenko normalize failed; returning original ({e})")
            return img


## Phase A — Dataset & data acquisition

In [ ]:
from PIL import Image

_IMG_EXTS = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}


class PathologyPatchDataset(Dataset):
    '''Reads class-named subfolders (ImageFolder layout). Albumentations-based transforms.'''

    def __init__(self, root, classes=None, transform=None, macenko=None):
        self.root = Path(root)
        if classes is None:
            classes = sorted([d.name for d in self.root.iterdir() if d.is_dir()])
        self.classes = classes
        self.class_to_idx = {c: i for i, c in enumerate(classes)}
        self.samples = []
        for c in classes:
            cdir = self.root / c
            if not cdir.is_dir():
                logger.warning(f"Missing class dir: {cdir}")
                continue
            for f in sorted(cdir.iterdir()):
                if f.suffix.lower() in _IMG_EXTS:
                    self.samples.append((str(f), self.class_to_idx[c]))
        if not self.samples:
            raise RuntimeError(f"No images found under {self.root}")
        self.transform = transform
        self.macenko = macenko
        logger.info(f"Dataset {self.root.name}: {len(self.samples)} imgs / {len(classes)} classes")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        try:
            img = np.array(Image.open(path).convert("RGB"))
        except Exception as e:
            logger.warning(f"Read failed {path} ({e}); skipping to next")
            return self.__getitem__((idx + 1) % len(self.samples))
        if self.macenko is not None:
            img = self.macenko(img)
        if self.transform is not None:
            img = self.transform(image=img)["image"]
        return img, label


In [ ]:
# Zenodo record 1214456 = NCT-CRC-HE-100K (train) + CRC-VAL-HE-7K (patient-disjoint val)
ZENODO = {
    "NCT-CRC-HE-100K": "https://zenodo.org/records/1214456/files/NCT-CRC-HE-100K.zip?download=1",
    "CRC-VAL-HE-7K":  "https://zenodo.org/records/1214456/files/CRC-VAL-HE-7K.zip?download=1",
}
# HuggingFace fallback (no token needed): use `datasets.load_dataset` mirrors if Zenodo is slow.


def _download(url, dest):
    logger.info(f"Downloading -> {dest}")
    last = [0]
    def hook(blocks, bs, total):
        if total > 0:
            pct = 100.0 * blocks * bs / total
            if pct - last[0] >= 2:
                last[0] = pct
                print(f"\r  {pct:5.1f}%", end="", flush=True)
    urllib.request.urlretrieve(url, dest, reporthook=hook)
    print()


def _find_class_root(base, name):
    '''Return the dir that actually contains the 9 class subfolders.'''
    cand = base / name
    if cand.is_dir() and any((cand / c).is_dir() for c in CLASSES):
        return cand
    for d in base.rglob("*"):
        if d.is_dir() and sum((d / c).is_dir() for c in CLASSES) >= 5:
            return d
    return cand


def ensure_dataset(persist_dir, which=("NCT-CRC-HE-100K", "CRC-VAL-HE-7K"), delete_zip=True):
    persist_dir = Path(persist_dir)
    data_dir = persist_dir / "data"
    data_dir.mkdir(parents=True, exist_ok=True)
    out = {}
    for name in which:
        root = _find_class_root(data_dir, name)
        if root.is_dir() and any((root / c).is_dir() for c in CLASSES):
            logger.info(f"{name} already present: {root}")
            out[name] = root
            continue
        zip_path = data_dir / f"{name}.zip"
        if not zip_path.exists():
            try:
                _download(ZENODO[name], zip_path)
            except Exception as e:
                raise RuntimeError(f"Download failed for {name} ({e}). "
                                   f"Manually place the extracted folder at {data_dir / name}") from e
        logger.info(f"Extracting {zip_path.name} ...")
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(data_dir)
        if delete_zip:
            zip_path.unlink(missing_ok=True)  # respect the shared-folder size ceiling
        out[name] = _find_class_root(data_dir, name)
        logger.info(f"{name} ready: {out[name]}")
    return out


def make_dataloaders(train_dir, val_dir, cfg):
    train_ds = PathologyPatchDataset(
        train_dir,
        transform=build_train_aug(cfg["use_stain_aug"], cfg["img_size"], cfg["hed_theta"]),
    )
    val_ds = PathologyPatchDataset(
        val_dir, classes=train_ds.classes, transform=build_eval_aug(cfg["img_size"]),
    )
    common = dict(num_workers=cfg["num_workers"], pin_memory=cfg["pin_memory"])
    if cfg["num_workers"] > 0:
        common.update(persistent_workers=True, prefetch_factor=cfg.get("prefetch_factor", 4))
    train_loader = DataLoader(train_ds, batch_size=cfg["batch_size"], shuffle=True,
                              drop_last=True, **common)
    val_loader = DataLoader(val_ds, batch_size=cfg["eval_batch_size"], shuffle=False, **common)
    return train_loader, val_loader, train_ds.classes


## Phase B — Model backbone strategy

* **Config 1 (full fine-tune):** `timm` ResNet-50 / ViT-B / ViT-L / Swin on ImageNet weights.
  Uses PyTorch SDPA attention (the portable flash/efficient kernel on ROCm).
* **Config 2 (frozen FM + MLP head):** `owkin/phikon` (un-gated) as a frozen feature extractor,
  CLS embedding -> trainable MLP head. Only the head is optimized.

In [ ]:
TIMM_MAP = {
    "resnet50": "resnet50.a1_in1k",
    "vit_b":    "vit_base_patch16_224.augreg2_in21k_ft_in1k",
    "vit_l":    "vit_large_patch16_224.augreg_in21k_ft_in1k",
    "swin_b":   "swin_base_patch4_window7_224.ms_in22k_ft_in1k",
}


class PhikonClassifier(nn.Module):
    '''Frozen Phikon backbone + trainable MLP head (Config 2).'''

    def __init__(self, num_classes=9, hidden=512, dropout=0.3, model_id="owkin/phikon"):
        super().__init__()
        _ensure("transformers")
        from transformers import AutoModel
        try:
            self.backbone = AutoModel.from_pretrained(model_id, attn_implementation="sdpa")
        except Exception as e:
            logger.warning(f"sdpa attn unavailable ({e}); using default attention")
            self.backbone = AutoModel.from_pretrained(model_id)
        for p in self.backbone.parameters():
            p.requires_grad_(False)
        self.backbone.eval()
        feat = self.backbone.config.hidden_size
        self.head = nn.Sequential(
            nn.LayerNorm(feat), nn.Linear(feat, hidden), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(hidden, num_classes),
        )

    def train(self, mode=True):
        super().train(mode)
        self.backbone.eval()  # backbone stays frozen / eval
        return self

    def forward(self, x):
        with torch.no_grad():
            out = self.backbone(pixel_values=x)
            cls = out.last_hidden_state[:, 0]
        return self.head(cls)


def build_model(cfg, num_classes=9):
    name = cfg["model"]
    if name == "phikon":
        return PhikonClassifier(num_classes=num_classes,
                                model_id=cfg.get("phikon_id", "owkin/phikon"))
    import timm
    if name not in TIMM_MAP:
        raise ValueError(f"Unknown model '{name}'. Options: {list(TIMM_MAP) + ['phikon']}")
    return timm.create_model(TIMM_MAP[name], pretrained=cfg.get("pretrained", True),
                             num_classes=num_classes)


def get_param_groups(model, cfg):
    '''Layer-wise LR: head LR > backbone LR for ViT/Swin; single group for ResNet/Phikon.'''
    name, lr, wd = cfg["model"], cfg["lr"], cfg["weight_decay"]
    if name == "phikon":
        return [{"params": [p for p in model.parameters() if p.requires_grad],
                 "lr": lr, "weight_decay": wd}]
    head_keys = ("head", "fc", "classifier")
    head, backbone = [], []
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        (head if any(k in n for k in head_keys) else backbone).append(p)
    groups = []
    if head:
        groups.append({"params": head, "lr": lr, "weight_decay": wd})
    if backbone:
        groups.append({"params": backbone, "lr": lr * cfg.get("backbone_lr_mult", 0.1),
                       "weight_decay": wd})
    return groups


## Helpers — metrics, Grad-CAM, resume-safe checkpoints

In [ ]:
from sklearn.metrics import (accuracy_score, f1_score, confusion_matrix,
                             classification_report)


@torch.no_grad()
def run_inference(model, loader, device, amp_dtype):
    model.eval()
    preds, labels = [], []
    use_amp = device.type == "cuda"
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        if use_amp:
            with torch.autocast(device_type="cuda", dtype=amp_dtype):
                out = model(x)
        else:
            out = model(x)
        preds.append(out.argmax(1).detach().cpu())
        labels.append(y)
    return torch.cat(preds).numpy(), torch.cat(labels).numpy()


def compute_metrics(labels, preds, classes=CLASSES):
    return {
        "accuracy": float(accuracy_score(labels, preds) * 100),
        "macro_f1": float(f1_score(labels, preds, average="macro") * 100),
        "per_class_f1": {c: float(v * 100) for c, v in
                         zip(classes, f1_score(labels, preds, average=None))},
    }


def plot_confusion_matrix(labels, preds, classes=CLASSES, save_path=None, normalize=True):
    import matplotlib.pyplot as plt
    import seaborn as sns
    cm = confusion_matrix(labels, preds, labels=list(range(len(classes))))
    if normalize:
        cm = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(min=1)
    fig, ax = plt.subplots(figsize=(9, 7))
    sns.heatmap(cm, annot=True, fmt=".2f" if normalize else "d", cmap="Blues",
                xticklabels=classes, yticklabels=classes, ax=ax, cbar=True)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title("Confusion Matrix — CRC-VAL-HE-7K")
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=180, bbox_inches="tight")
        logger.info(f"Saved confusion matrix -> {save_path}")
    return fig


In [ ]:
class _GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.acts = None
        self.grads = None
        self._h = [target_layer.register_forward_hook(self._fwd),
                   target_layer.register_full_backward_hook(self._bwd)]

    def _fwd(self, m, i, o):
        self.acts = o

    def _bwd(self, m, gi, go):
        self.grads = go[0]

    def remove(self):
        for h in self._h:
            h.remove()


class GradCAMViT(_GradCAM):
    def __init__(self, model, target_layer=None):
        super().__init__(model, target_layer or model.blocks[-1].norm1)

    def __call__(self, x, class_idx=None):
        self.model.eval()
        out = self.model(x)
        class_idx = out.argmax(1).item() if class_idx is None else class_idx
        self.model.zero_grad(set_to_none=True)
        out[0, class_idx].backward()
        acts, grads = self.acts[0][1:], self.grads[0][1:]   # drop CLS token
        cam = (acts * grads.mean(0)).sum(-1).clamp(min=0)
        s = int(cam.shape[0] ** 0.5)
        cam = cam.reshape(s, s).detach().cpu().numpy()
        return _norm_resize(cam, x.shape[-1]), class_idx


class GradCAMResNet(_GradCAM):
    def __init__(self, model, target_layer=None):
        super().__init__(model, target_layer or model.layer4[-1])

    def __call__(self, x, class_idx=None):
        self.model.eval()
        out = self.model(x)
        class_idx = out.argmax(1).item() if class_idx is None else class_idx
        self.model.zero_grad(set_to_none=True)
        out[0, class_idx].backward()
        acts, grads = self.acts[0], self.grads[0]           # (C,H,W)
        cam = (acts * grads.mean((1, 2))[:, None, None]).sum(0).clamp(min=0)
        return _norm_resize(cam.detach().cpu().numpy(), x.shape[-1]), class_idx


def _norm_resize(cam, size):
    cam = cam - cam.min()
    cam = cam / (cam.max() + 1e-8)
    return cv2.resize(cam, (size, size))


def make_gradcam(model, model_name):
    if model_name == "resnet50":
        return GradCAMResNet(model)
    if model_name in ("vit_b", "vit_l"):
        return GradCAMViT(model)
    logger.warning(f"Grad-CAM not wired for '{model_name}'; returning None")
    return None


def overlay_heatmap(image_rgb, cam, alpha=0.5):
    heat = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)
    heat = cv2.cvtColor(heat, cv2.COLOR_BGR2RGB)
    return cv2.addWeighted(np.asarray(image_rgb, dtype=np.uint8), 1 - alpha, heat, alpha, 0)


def denormalize(tensor):
    '''CHW normalized tensor -> HxWx3 uint8 RGB for visualization / overlay.'''
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    img = (tensor.detach().cpu() * std + mean).clamp(0, 1)
    return (img.permute(1, 2, 0).numpy() * 255).astype(np.uint8)


In [ ]:
def save_checkpoint(state, path):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(".tmp")
    torch.save(state, tmp)
    tmp.replace(path)  # atomic write -> survives a killed kernel mid-save


def load_checkpoint(path, map_location="cpu"):
    return torch.load(path, map_location=map_location, weights_only=False)


def raw_module(model):
    '''Unwrap torch.compile so checkpoints have clean (prefix-free) state_dict keys.'''
    return getattr(model, "_orig_mod", model)


print("shared.ipynb loaded: Phase A + B + helpers ready.")
